# ⚽ Dal Web all'App: Mondiale 2026

Fino ad ora vi ho sempre dato io i dati (file CSV da Kaggle).

Oggi **ve li prendete da soli**, direttamente da internet.

**Cosa faremo:**
1. Prendere i dati del Mondiale 2026 da Wikipedia (web scraping)
2. Pulirli e analizzarli (pandas)
3. Creare grafici (matplotlib)
4. Trasformare tutto in un'app interattiva (Streamlit)

## 1. Web Scraping: prendere dati dal web

**Web scraping** = estrarre dati da una pagina web in modo automatico.

Wikipedia ha tantissime tabelle HTML. Con `pandas.read_html()` possiamo leggerle
e trasformarle direttamente in DataFrame — con **2 righe di codice**.

Fonte: [2026 FIFA World Cup - Wikipedia](https://en.wikipedia.org/wiki/2026_FIFA_World_Cup)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# --- WEB SCRAPING ---
# requests.get() scarica il contenuto HTML di una pagina web
# headers: Wikipedia blocca le richieste senza User-Agent (una stringa
# che identifica il programma che fa la richiesta)
import requests
from io import StringIO

url = 'https://en.wikipedia.org/wiki/2026_FIFA_World_Cup'
headers = {'User-Agent': 'Mozilla/5.0 (educational project)'}
response = requests.get(url, headers=headers)

print(f'Status: {response.status_code}')  # 200 = tutto ok
print(f'Dimensione pagina: {len(response.text):,} caratteri')

In [ ]:
# pd.read_html() legge TUTTE le tabelle HTML dalla pagina
# Restituisce una lista di DataFrame (una per ogni tabella trovata)
# StringIO serve a "fingere" che il testo HTML sia un file
tables = pd.read_html(StringIO(response.text))

print(f'Tabelle trovate nella pagina: {len(tables)}')
print()
print('Questa è la potenza del web scraping:')
print(f'Con 2 righe di codice abbiamo estratto {len(tables)} tabelle da Wikipedia!')

## 2. Troviamo le tabelle dei gironi

Abbiamo ~150 tabelle, ma ci servono solo quelle dei **gironi** (Group A, B, C...).

Come le riconosciamo? Hanno tutte la stessa forma: **4 righe** (4 squadre) e **11 colonne** (Pos, Team, Pld, W, D, L, GF, GA, GD, Pts, Qualification).

In [ ]:
# Cerchiamo tutte le tabelle con esattamente 4 righe e 11 colonne
# Questo è un pattern comune nel web scraping:
# "non so QUALE tabella mi serve, ma so COM'È FATTA"

group_indices = [i for i, t in enumerate(tables) if t.shape == (4, 11)]
print(f'Tabelle dei gironi trovate: {len(group_indices)}')
print(f'Indici: {group_indices}')
print()

# Guardiamone una per capire la struttura
print('Esempio - Girone A:')
tables[group_indices[0]]

## 3. Costruiamo il dataset completo

Uniamo tutte le tabelle dei gironi in un unico DataFrame con una colonna `Group`.

In [ ]:
# Lettere dei gironi: A, B, C, ... L (12 gironi nel Mondiale 2026)
group_letters = list('ABCDEFGHIJKL')

# Per ogni tabella, aggiungiamo la colonna 'Group' e la mettiamo in una lista
groups = []
for idx, table_idx in enumerate(group_indices):
    t = tables[table_idx].copy()
    t['Group'] = group_letters[idx]
    groups.append(t)

# pd.concat() impila i DataFrame uno sotto l'altro
# ignore_index=True: rinumera le righe da 0
df = pd.concat(groups, ignore_index=True)

print(f'Dataset completo: {len(df)} squadre, {len(df.columns)} colonne')
df.head(10)

## 4. Puliamo i dati

I dati dal web non sono mai puliti. Guardate la colonna `Teamvte`:
ha testo in più come `(H)` per le squadre ospitanti. Puliamo.

In [ ]:
# Rinominiamo la colonna 'Teamvte' in 'Team' (più chiaro)
df = df.rename(columns={'Teamvte': 'Team'})

# Puliamo il nome delle squadre:
# .str è l'accessor per operazioni sulle stringhe
# .replace() sostituisce un pattern con un altro
# regex=True: usa espressioni regolari (pattern avanzati)
# r'\s*\(H\)': qualsiasi spazio + (H)
# r'\[.*\]': qualsiasi cosa tra parentesi quadre [a], [b], etc.
df['Team'] = df['Team'].str.replace(r'\s*\(H\)', '', regex=True)
df['Team'] = df['Team'].str.strip()  # toglie spazi extra all'inizio e alla fine

# Convertiamo Pts in numerico (a volte ha annotazioni come '4[a]')
df['Pts'] = pd.to_numeric(df['Pts'].astype(str).str.replace(r'\[.*\]', '', regex=True))

# Convertiamo GD (goal difference) in numerico
# Il carattere '−' (minus Unicode) va sostituito con '-' (minus ASCII)
df['GD'] = pd.to_numeric(df['GD'].astype(str).str.replace('−', '-'))

print('Dataset pulito:')
df[['Team', 'Group', 'Pld', 'W', 'D', 'L', 'GF', 'GA', 'GD', 'Pts']].head(10)

In [ ]:
df.describe()

## 5. Grafici: cosa ci raccontano i dati?

### Le squadre con più gol fatti

In [ ]:
# Ordiniamo per gol fatti (GF) in ordine decrescente e prendiamo le top 10
# .sort_values(): ordina il DataFrame
# ascending=False: dal più grande al più piccolo
top_scorers = df.sort_values('GF', ascending=False).head(10)

plt.figure(figsize=(10, 6))

# plt.barh() crea un grafico a barre ORIZZONTALI
# È meglio del verticale quando i nomi sono lunghi
plt.barh(top_scorers['Team'], top_scorers['GF'], color='steelblue')

plt.xlabel('Gol fatti', fontsize=13)
plt.title('Top 10 squadre per gol fatti - Mondiale 2026', fontsize=14)
plt.gca().invert_yaxis()  # la prima in alto, non in basso
plt.tight_layout()
plt.show()

### Gol fatti vs gol subiti: chi attacca e chi difende?

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df['GF'], df['GA'], s=80, alpha=0.7, color='darkorange')

# Aggiungiamo il nome di ogni squadra accanto al punto
# plt.annotate() mette un testo in una posizione specifica del grafico
for _, row in df.iterrows():
    plt.annotate(row['Team'], (row['GF'], row['GA']),
                 fontsize=7, alpha=0.7,
                 xytext=(5, 5), textcoords='offset points')

plt.xlabel('Gol fatti (GF)', fontsize=13)
plt.ylabel('Gol subiti (GA)', fontsize=13)
plt.title('Attacco vs Difesa - Mondiale 2026', fontsize=14)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# COME LEGGERE IL GRAFICO:
# In basso a destra = tanti gol fatti, pochi subiti (squadra forte)
# In alto a sinistra = pochi gol fatti, tanti subiti (squadra debole)

### Punti medi per girone: qual è il girone di ferro?

In [ ]:
# Media punti per girone
avg_pts = df.groupby('Group')['Pts'].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
avg_pts.plot(kind='bar', color='green', alpha=0.8)
plt.xlabel('Girone', fontsize=13)
plt.ylabel('Punti medi', fontsize=13)
plt.title('Punti medi per girone - Mondiale 2026', fontsize=14)
plt.xticks(rotation=0)  # lettere dei gironi dritte, non inclinate
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

## 6. Salviamo il dataset

Abbiamo costruito un dataset dal nulla. Salviamolo come CSV per usarlo dopo.

In [ ]:
# Salva il DataFrame come CSV
# index=False: non salvare il numero di riga
df.to_csv('mondiale_2026_gironi.csv', index=False)

# Su Colab: scarica il file sul tuo PC
from google.colab import files
files.download('mondiale_2026_gironi.csv')

print('File salvato! Questo CSV lo useremo per l\'app Streamlit.')

## 7. Riepilogo: il percorso completo

```
Wikipedia (HTML)                  
       |
  requests.get()          ← scarica la pagina
       |
  pd.read_html()          ← estrae le tabelle
       |
  concat + pulizia        ← costruisce il dataset
       |
  scatter + bar chart     ← analizza e visualizza
       |
  .to_csv()               ← salva il risultato
       |
  Streamlit               ← trasforma in app!
```

**Prossimo step:** apriamo GitHub Codespaces e trasformiamo questi grafici in un'app interattiva.